# 文書から質問と答えの組を自動で作る（質問生成）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_qa/question_generation.ipynb)

NLP クックブック 22-08 ／ 質問応答

文書を読んで答えられる質問を自動で作りたい。QA や検索の評価セット、理解度を確かめる練習問題に使いたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import re
import sentencepiece as spm
import torch
from fugashi import Tagger
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoModelForQuestionAnswering, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def chat(messages, max_new_tokens=60):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

passage = ("琵琶湖は、滋賀県にある日本最大の湖である。面積は約 670 平方キロメートルで、滋賀県の面積のおよそ 6 分の 1 を占める。"
           "約 400 万年前に誕生したとされ、世界でも有数の古代湖である。湖から流れ出る川は瀬田川だけで、下流では淀川と名前を変えて大阪湾に注ぐ。"
           "近畿地方の約 1450 万人の生活用水を支えており、「近畿の水がめ」と呼ばれる。")

# 1. 答えの候補を選ぶ: 数値を含む表現と固有名詞を、文書から機械的に取り出す
tagger = Tagger()
candidates = re.findall(r"約?\s*\d[\d,.]*\s*(?:万|億)?\s*(?:平方キロメートル|キロメートル|万年前|年前|分の\s*\d+|万人|人|年)", passage)
buf = []
for w in list(tagger(passage)) + [None]:
    is_proper = w is not None and w.feature.pos1 == "名詞" and w.feature.pos2 == "固有名詞"
    is_tail = w is not None and buf and w.feature.pos1 in ("名詞", "接尾辞") and w.feature.pos2 != "数詞"   # 琵琶 + 湖、滋賀 + 県 のように後ろに続く語
    if is_proper or is_tail:
        buf.append(w.surface)
    else:
        if buf:
            candidates.append("".join(buf))
        buf = []
candidates = list(dict.fromkeys(c.strip() for c in candidates))
print(candidates)

['約 670 平方キロメートル', '6 分の 1', '約 400 万年前', '約 1450 万人', '琵琶湖', '滋賀県', '日本最大', '瀬田川', '淀川', '大阪湾', '近畿地方', '近畿']


In [3]:
# 2. 答えを指定して、それが答えになる質問を作らせる（例を 1 つ見せる）
INSTRUCTION = "次の文章を読めば答えられる質問を 1 つ作ってください。質問の答えは、指定された「答え」になるようにしてください。質問文だけを出力してください。"
SHOT = [{"role": "user", "content": f"{INSTRUCTION}\n\n# 文章\n富士山は、静岡県と山梨県にまたがる活火山である。標高は 3776 メートルで、日本の最高峰である。\n\n# 答え\n3776 メートル"},
        {"role": "assistant", "content": "富士山の標高は何メートルですか？"}]

def generate_question(passage, answer):
    return chat(SHOT + [{"role": "user", "content": f"{INSTRUCTION}\n\n# 文章\n{passage}\n\n# 答え\n{answer}"}]).split("\n")[0]

pairs = [(generate_question(passage, a), a) for a in candidates[:8]]
for q, a in pairs:
    print(f"Q: {q}\n   A: {a}")

Q: 琵琶湖の面積は約何平方キロメートルですか？
   A: 約 670 平方キロメートル
Q: 琵琶湖は滋賀県の面積の約何分の何を占めていますか？
   A: 6 分の 1
Q: 琵琶湖が約400万年前に誕生したという情報に基づいて、琵琶湖よりも最近誕生した日本の湖で、有名なものを一つ挙げてください。
   A: 約 400 万年前
Q: 琵琶湖は近畿地方のどのくらいの人口の生活用水を供給していますか？
   A: 約 1450 万人
Q: 琵琶湖は滋賀県に所在し、日本最大の湖ですが、その面積は滋賀県の面積の何分の何を占めていますか？
   A: 琵琶湖
Q: 琵琶湖はどの都道府県に位置していますか？
   A: 滋賀県
Q: 琵琶湖は日本で何番目に大きい湖ですか？
   A: 日本最大
Q: 琵琶湖から流れ出る川の名前は何といいますか？
   A: 瀬田川


In [4]:
# 3. 往復検査: 作った質問を QA モデルに解かせて、指定した答えが返ってくるものだけを残す
qa_name = "tsmatz/roberta_qa_japanese"
qa_tok = AutoTokenizer.from_pretrained(qa_name)
# transformers 5 系の T5Tokenizer は未知語を文末記号 (</s>) の ID にしてしまうので、Unigram のモデルを組み立て直す（この章の「文書の中から質問の答えを抜き出す（抽出型 QA）」を参照）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(qa_name, "spiece.model"))
qa_tok.backend_tokenizer.model = models.Unigram([(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id())
qa = AutoModelForQuestionAnswering.from_pretrained(qa_name).eval()

@torch.no_grad()
def extract(question, context):
    enc = qa_tok(question, context, return_tensors="pt", truncation="only_second", max_length=512, return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]
    out = qa(**enc)
    in_context = torch.tensor([s == 1 for s in enc.sequence_ids(0)])
    scores = out.start_logits[0][:, None] + out.end_logits[0][None, :]
    scores = scores.masked_fill(~in_context[:, None] | ~in_context[None, :], -1e9)
    scores = scores.triu().masked_fill(torch.ones_like(scores).tril(-1).bool(), -1e9).masked_fill(torch.ones_like(scores).triu(30).bool(), -1e9)
    s, e = divmod(scores.argmax().item(), scores.shape[1])
    return context[offsets[s][0]:offsets[e][1]]

squash = lambda s: re.sub(r"\s|約", "", s)
kept = []
for q, a in pairs:
    predicted = extract(q, passage)
    ok = squash(a) in squash(predicted) or squash(predicted) in squash(a)
    kept += [(q, a)] if ok and predicted else []
    print(f"{'○' if ok else '×'} {q} → QA モデルの答え: {predicted!r} (指定: {a!r})")
print(len(kept), "/", len(pairs), "組を採用")

○ 琵琶湖の面積は約何平方キロメートルですか？ → QA モデルの答え: '約 670 平方キロメートル' (指定: '約 670 平方キロメートル')
○ 琵琶湖は滋賀県の面積の約何分の何を占めていますか？ → QA モデルの答え: 'およそ 6 分の 1 を' (指定: '6 分の 1')
× 琵琶湖が約400万年前に誕生したという情報に基づいて、琵琶湖よりも最近誕生した日本の湖で、有名なものを一つ挙げてください。 → QA モデルの答え: '琵琶湖' (指定: '約 400 万年前')
○ 琵琶湖は近畿地方のどのくらいの人口の生活用水を供給していますか？ → QA モデルの答え: 'の約 1450 万人の' (指定: '約 1450 万人')
× 琵琶湖は滋賀県に所在し、日本最大の湖ですが、その面積は滋賀県の面積の何分の何を占めていますか？ → QA モデルの答え: 'およそ 6 分の 1 を' (指定: '琵琶湖')
○ 琵琶湖はどの都道府県に位置していますか？ → QA モデルの答え: '滋賀県' (指定: '滋賀県')
○ 琵琶湖は日本で何番目に大きい湖ですか？ → QA モデルの答え: '日本' (指定: '日本最大')
○ 琵琶湖から流れ出る川の名前は何といいますか？ → QA モデルの答え: '瀬田川' (指定: '瀬田川')
6 / 8 組を採用


- 「答えを先に決めてから質問を作らせる」と、答えが文書中にあることが保証され、抽出型 QA のデータとしてそのまま使える。
- 生成した質問には、答えが一意に決まらないもの、文書を読まなくても答えられるもの、指定した答えと対応しないものが混ざる。QA モデルで解き直して答えが一致するものだけを残す（往復検査）と、質がかなり揃う。
- 上の往復検査は「一方が他方を含む」で判定しているので、「日本最大」を答えに指定した「日本で何番目に大きい湖ですか？」のように、QA モデルが「日本」とだけ返した組も残ってしまう。厳しくするなら、文字単位の F1 にしきい値を設ける。
- RAG や検索の評価セットを作るときにも同じ方法が使える。ただし、文書の言い回しをなぞった質問になりやすく、実際のユーザーの質問より易しい点に注意する。
- 練習問題として使うなら、答えの候補の選び方が出題の質を決める。重要な概念を問うには、候補を人が選ぶか、重要語の抽出と組み合わせる。